# Jothidam / Matrimony Custom LLM — Fine-Tuning Notebook

This notebook fine-tunes an open-source LLM (Qwen2.5-7B-Instruct) on your jothidam/matrimony domain data using QLoRA (via Unsloth), so it behaves like a domain-specific model trained on your field only.

**Run this on Google Colab with a free T4 GPU** (Runtime > Change runtime type > T4 GPU).

Steps: Install libs -> Load base model -> Load your training_data.jsonl -> Fine-tune (LoRA) -> Save -> Test.

## 1. Install dependencies (Colab)

In [ ]:
!pip install -q unsloth
!pip install -q --no-deps trl peft accelerate bitsandbytes

## 2. Load base model (4-bit, QLoRA)

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048

# Qwen2.5-7B-Instruct: strong multilingual (Tamil+English) support, ungated, free to use.
# Alternatives: 'unsloth/llama-3-8b-Instruct-bnb-4bit' (needs HF gated access approval),
#               'unsloth/mistral-7b-instruct-v0.3-bnb-4bit'
model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 3. Upload and load your training data
Upload `training_data.jsonl` to the Colab file panel (or mount Google Drive), then run below.

**Important:** Add 200-1000+ examples for good results. Include some 'refuse off-topic' examples (like the sample) so the model learns to stay within your field.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files="training_data.jsonl", split="train")

SYSTEM_PROMPT = (
    "You are a jothidam (astrology) and matrimony matching assistant. "
    "Only answer questions about horoscope matching, Porutham, dosham, Nakshatra, Rasi, "
    "and matrimony/profile matching. If asked anything outside this field, politely say "
    "it is outside your area and redirect to jothidam/matrimony topics."
)

def format_prompt(example):
    text = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{example['instruction']}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['output']}<|im_end|>"
    )
    return {"text": text}

dataset = dataset.map(format_prompt)
print(dataset[0]["text"])

## 4. Fine-tune (SFTTrainer)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## 5. Save the fine-tuned model

In [ ]:
model.save_pretrained("jothidam_llm_lora")
tokenizer.save_pretrained("jothidam_llm_lora")

# Optional: merge LoRA into base model for a standalone model, and save 16-bit merged version
model.save_pretrained_merged("jothidam_llm_merged", tokenizer, save_method="merged_16bit")

# Optional: push to your own private Hugging Face Hub repo for easy deployment
# model.push_to_hub_merged("your-username/jothidam-llm", tokenizer, save_method="merged_16bit", token="YOUR_HF_TOKEN")

## 6. Quick test

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question):
    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{question}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)
    print(tokenizer.decode(outputs[0], skip_special_tokens=True))

ask("What is Rajju Dosham?")
ask("Recommend a good movie to watch")  # should politely refuse - off-topic